In [ ]:
cd /content/drive/MyDrive/projects/Blockchain/trash

/content/drive/MyDrive/projects/Blockchain/trash


In [ ]:
!pwd

/content/drive/MyDrive/projects/Blockchain/trash


In [ ]:
import os
import argparse
import copy
import time
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
import numpy as np

In [ ]:
IMAGE_SIZE   = 224
BATCH_SIZE   = 32
NUM_EPOCHS   = 60
LR           = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE     = 7       # early stopping patience (epochs)
NUM_WORKERS  = 4       # dataloader workers (set 0 on Windows)

In [ ]:
def get_transforms():
    """
    Training:   resize → random crop/flip → CCTV augmentation → normalize
    Validation: resize → center crop → normalize (no augmentation)
    """
    imagenet_mean = [0.485, 0.456, 0.406]
    imagenet_std  = [0.229, 0.224, 0.225]

    train_tf = transforms.Compose([
        # ── Resize & spatial augmentation
        transforms.Resize((256, 256)),
        transforms.RandomCrop(IMAGE_SIZE),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.1),
        transforms.RandomRotation(degrees=15),

        # ── CCTV simulation
        transforms.ColorJitter(
            brightness=0.4,    # lighting variation (day/night/IR)
            contrast=0.4,      # washed out / overexposed
            saturation=0.3,    # color degradation
            hue=0.1
        ),
        transforms.RandomGrayscale(p=0.2),          # IR / night vision mode
        transforms.GaussianBlur(kernel_size=3,
                                sigma=(0.1, 2.0)),   # motion blur / low res

        # ── To tensor + normalize
        transforms.ToTensor(),
        transforms.Normalize(mean=imagenet_mean,
                             std=imagenet_std),

        # ── Sensor noise (applied after ToTensor)
        transforms.Lambda(
            lambda x: x + 0.015 * torch.randn_like(x)
        ),
    ])

    val_tf = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),   # direct resize, no crop
        transforms.ToTensor(),
        transforms.Normalize(mean=imagenet_mean,
                             std=imagenet_std),
    ])

    return train_tf, val_tf

In [ ]:
def load_datasets(data_root, batch_size, num_workers):
    """
    Load train and val splits from data_root.
    Computes class weights from training set for weighted loss.

    Returns:
        dataloaders : dict with 'train' and 'val' DataLoaders
        class_names : list e.g. ['no_trash', 'trash']
        class_weights: Tensor for weighted CrossEntropyLoss
    """
    train_tf, val_tf = get_transforms()

    train_dataset = datasets.ImageFolder(
        root=os.path.join(data_root, "train"),
        transform=train_tf
    )
    val_dataset = datasets.ImageFolder(
        root=os.path.join(data_root, "val"),
        transform=val_tf
    )

    # ── Class weights (inverse frequency) to handle imbalance
    class_counts  = np.bincount(train_dataset.targets)
    total_samples = len(train_dataset.targets)
    class_weights = torch.tensor(
        [total_samples / (len(class_counts) * c) for c in class_counts],
        dtype=torch.float
    )

    dataloaders = {
        "train": DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=num_workers,
            pin_memory=True
        ),
        "val": DataLoader(
            val_dataset,
            batch_size=batch_size,
            shuffle=False,
            num_workers=num_workers,
            pin_memory=True
        ),
    }

    print(f"\n Dataset summary:")
    print(f"     Classes     : {train_dataset.classes}")
    print(f"     Train images: {len(train_dataset):,}")
    print(f"     Val images  : {len(val_dataset):,}")
    print(f"     Class counts (train): { {train_dataset.classes[i]: int(c) for i, c in enumerate(class_counts)} }")
    print(f"     Class weights       : { {train_dataset.classes[i]: f'{w:.3f}' for i, w in enumerate(class_weights)} }")

    return dataloaders, train_dataset.classes, class_weights

In [ ]:
def build_model(num_classes=2):
    """
    MobileNetV3-Large with ImageNet pretrained weights.
    Replaces the classifier head for binary classification.
    """
    model = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)

    # Freeze all backbone layers initially
    for param in model.parameters():
        param.requires_grad = False

    # Replace classifier head — this is the only part trained initially
    in_features = model.classifier[0].in_features
    model.classifier = nn.Sequential(
        nn.Linear(in_features, 1280),
        nn.Hardswish(),
        nn.Dropout(p=0.2),
        nn.Linear(1280, num_classes)
    )

    # Classifier head params are trainable by default (requires_grad=True)
    return model

In [ ]:
def unfreeze_backbone(model, unfreeze_from_layer=14):
    """
    Unfreeze later backbone layers for fine-tuning after initial warmup.
    MobileNetV3-Large has 16 feature blocks (features[0..15]).
    Unfreezing from layer 14 means the last 2 blocks + classifier are trained.
    """
    for i, layer in enumerate(model.features):
        if i >= unfreeze_from_layer:
            for param in layer.parameters():
                param.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"\n  Backbone partially unfrozen (from layer {unfreeze_from_layer})")
    print(f"     Trainable params: {trainable:,} / {total:,}")

In [ ]:
def train_model(model, dataloaders, criterion, optimizer, scheduler,
                device, num_epochs, patience, output_dir):
    """
    Full training loop with:
        - Phase 1 (epochs 1–5):  train classifier head only (backbone frozen)
        - Phase 2 (epoch 6+):    unfreeze last 2 backbone blocks + fine-tune
        - Early stopping on val loss
        - Save best model checkpoint
    """
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    best_model_path = os.path.join(output_dir, "best_model.pth")

    best_val_acc  = 0.0
    best_val_loss = float("inf")
    best_weights  = copy.deepcopy(model.state_dict())
    no_improve    = 0
    history       = {"train_loss": [], "val_loss": [],
                     "train_acc":  [], "val_acc":  []}

    print("\n" + "=" * 60)
    print("  Training MobileNetV3 — Trash Binary Classifier")
    print("=" * 60)

    for epoch in range(1, num_epochs + 1):

        # ── Phase 2: unfreeze backbone after warmup
        if epoch == 6:
            unfreeze_backbone(model, unfreeze_from_layer=14)
            # Lower LR for fine-tuning
            for g in optimizer.param_groups:
                g["lr"] = g["lr"] * 0.1
            print(f"  LR reduced for fine-tuning: {optimizer.param_groups[0]['lr']:.2e}")

        epoch_start = time.time()
        print(f"\n  Epoch {epoch}/{num_epochs}  {'(warmup)' if epoch < 6 else '(fine-tune)'}")
        print(f"  {'─' * 50}")

        for phase in ["train", "val"]:
            model.train() if phase == "train" else model.eval()

            running_loss    = 0.0
            running_correct = 0
            total_samples   = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == "train"):
                    outputs = model(inputs)
                    loss    = criterion(outputs, labels)
                    preds   = outputs.argmax(dim=1)

                    if phase == "train":
                        loss.backward()
                        optimizer.step()

                running_loss    += loss.item() * inputs.size(0)
                running_correct += (preds == labels).sum().item()
                total_samples   += inputs.size(0)

            epoch_loss = running_loss    / total_samples
            epoch_acc  = running_correct / total_samples

            history[f"{phase}_loss"].append(epoch_loss)
            history[f"{phase}_acc"].append(epoch_acc)

            print(f"  {phase.upper():5s}  loss: {epoch_loss:.4f}  acc: {epoch_acc:.4f}")

            # ── Save best model (based on val accuracy)
            if phase == "val":
                if epoch_acc > best_val_acc or (
                    epoch_acc == best_val_acc and epoch_loss < best_val_loss
                ):
                    best_val_acc  = epoch_acc
                    best_val_loss = epoch_loss
                    best_weights  = copy.deepcopy(model.state_dict())
                    torch.save({
                        "epoch":       epoch,
                        "model_state": best_weights,
                        "val_acc":     best_val_acc,
                        "val_loss":    best_val_loss,
                        "class_names": dataloaders["train"].dataset.classes,
                    }, best_model_path)
                    print(f"  Best model saved  (val_acc={best_val_acc:.4f})")
                    no_improve = 0
                else:
                    no_improve += 1

        # ── Scheduler step (after val)
        if epoch >= 6:
            scheduler.step()

        elapsed = time.time() - epoch_start
        print(f"   Epoch time: {elapsed:.1f}s")

        # ── Early stopping
        if no_improve >= patience:
            print(f"\n   Early stopping triggered (no improvement for {patience} epochs)")
            break

    # Restore best weights
    model.load_state_dict(best_weights)

    print("\n" + "=" * 60)
    print(f"     Training complete")
    print(f"     Best val accuracy : {best_val_acc:.4f}")
    print(f"     Best val loss     : {best_val_loss:.4f}")
    print(f"     Model saved to    : {best_model_path}")
    print("=" * 60)

    return model, history

In [ ]:
parser = argparse.ArgumentParser(
        description="Train MobileNetV3 binary classifier for trash detection"
    )
parser.add_argument("--data",        required=True,
                    help="Path to dataset root (contains train/ and val/)")
parser.add_argument("--output",      default="checkpoints",
                    help="Directory to save model checkpoints (default: checkpoints/)")
parser.add_argument("--epochs",      type=int,   default=NUM_EPOCHS)
parser.add_argument("--batch_size",  type=int,   default=BATCH_SIZE)
parser.add_argument("--lr",          type=float, default=LR)
parser.add_argument("--patience",    type=int,   default=PATIENCE)
parser.add_argument("--num_workers", type=int,   default=NUM_WORKERS)
# args = parser.parse_args()
args = argparse.Namespace(
    data="/content/drive/MyDrive/projects/Blockchain/trash/output_dataset",
    output="checkpoints/",
    epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    patience=PATIENCE,
    num_workers=NUM_WORKERS,
)

# ── Device
device = (
    torch.device("cuda")  if torch.cuda.is_available()  else
    torch.device("mps")   if torch.backends.mps.is_available() else
    torch.device("cpu")
)
print(f"\n   Device: {device}")

# ── Data
dataloaders, class_names, class_weights = load_datasets(
    args.data, args.batch_size, args.num_workers
)

# ── Model
model = build_model(num_classes=len(class_names)).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"\n  MobileNetV3-Large loaded")
print(f"     Trainable params : {trainable:,} / {total:,}  (head only — warmup phase)")

# ── Loss with class weights (handles imbalance)
criterion = nn.CrossEntropyLoss(
    weight=class_weights.to(device)
)

# ── Optimizer (only classifier head params initially)
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=args.lr,
    weight_decay=WEIGHT_DECAY
)

# ── LR scheduler (cosine annealing — kicks in during fine-tune phase)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=args.epochs - 5, eta_min=1e-6
)

# ── Train
model, history = train_model(
    model, dataloaders, criterion, optimizer, scheduler,
    device, args.epochs, args.patience, args.output
)

# ── Save final training history
history_path = os.path.join(args.output, "history.txt")
with open(history_path, "w") as f:
    f.write("epoch,train_loss,val_loss,train_acc,val_acc\n")
    for i in range(len(history["train_loss"])):
        f.write(f"{i+1},"
                f"{history['train_loss'][i]:.4f},"
                f"{history['val_loss'][i]:.4f},"
                f"{history['train_acc'][i]:.4f},"
                f"{history['val_acc'][i]:.4f}\n")
print(f"\n  Training history saved to: {history_path}")
print(f"\n  Class index mapping: { {i: c for i, c in enumerate(class_names)} }")
print("\n  Next step: run inference.py to test on new images.\n")


   Device: cuda

 Dataset summary:
     Classes     : ['no_trash', 'trash']
     Train images: 15,963
     Val images  : 542
     Class counts (train): {'no_trash': 7333, 'trash': 8630}
     Class weights       : {'no_trash': '1.088', 'trash': '0.925'}

  MobileNetV3-Large loaded
     Trainable params : 1,232,642 / 4,204,594  (head only — warmup phase)

  Training MobileNetV3 — Trash Binary Classifier

  Epoch 1/60  (warmup)
  ──────────────────────────────────────────────────
  TRAIN  loss: 0.1452  acc: 0.9448
  VAL    loss: 0.1172  acc: 0.9502
  Best model saved  (val_acc=0.9502)
   Epoch time: 288.1s

  Epoch 2/60  (warmup)
  ──────────────────────────────────────────────────
  TRAIN  loss: 0.1083  acc: 0.9586
  VAL    loss: 0.0838  acc: 0.9705
  Best model saved  (val_acc=0.9705)
   Epoch time: 215.8s

  Epoch 3/60  (warmup)
  ──────────────────────────────────────────────────
  TRAIN  loss: 0.0980  acc: 0.9634
  VAL    loss: 0.1057  acc: 0.9613
   Epoch time: 222.5s

  Epoch 4/60